# Introduction to the Dataset

The dataset used in this tutorial is the German national forest inventory dataset available [here](https://www.openagrar.de/receive/openagrar_mods_00094435).

Parts of this notebook are also taken from the original notebook provided by the data creaters [here](https://www.openagrar.de/servlets/MCRFileNodeServlet/openagrar_derivate_00058265/intro_to_dataset.ipynb)

For reading about the dataset, please refer to the article [A Sentinel-2 machine learning dataset for tree species classification in Germany](https://essd.copernicus.org/articles/17/351/2025/)

The dataset contains the bottom of atmosphere reflectance values for the majority of the trees in the 2012 German national forest inventory.

The attributes of this dataset are:

![attributes](https://essd.copernicus.org/articles/17/351/2025/essd-17-351-2025-t01-thumb.png)

In [2]:
import math
import sqlite3
import datetime
import numpy as np
import pandas as pd
import polars as pl
from matplotlib import pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
import geopandas as gpd

## Data Loading & Formatting

The dataset comes in the form of an sqlite file. An SQLite file is a single, self-contained, cross-platform disk file that holds an entire *SQLite database*. We first create an sqlite connection and tell sqlite to return strings as byte-arrays. The data is stored in the sqlite table "data".

In [3]:
dataset_path = "S2GNFI_V1" # change this path to point to the database file after you download and unzip it from the source given above
conn = sqlite3.connect(dataset_path)
conn.text_factory = bytes  # this makes sqlite return strings as bytes that we can parse via numpy

To load the entire dataset, we would need ~30 GB of RAM!
But thanks to sqlite we can limit loading to a subset just to inspect a subset of the data. 
For the sake of this tutorial, we restrict loading to three classes of trees, Norway spruce (identified by integer 10), Scots pine (identified by integer 20) and beech (identified by integer 100). We add some more filters to keep the most-recently verified data only, and we keep only features that seem relevant to the task of tree classification.

The following piece of code loads a portion of the sqlite database that passes through our filters set in the "query" into a a two-dimensional tabular data structure, called a DataFrame.

In [4]:
query = """
SELECT
    species,
    boa,
    qai,
    dbh_mm,
    height_dm,
    crown_area_m2,
    time,
    tree_id,
    X_WGS84,
    Y_WGS84
FROM data
WHERE species IN (10, 100, 20)
  AND present_2022 = 1
  AND is_corrected = 1
  AND is_pure = 1
"""
df = pd.read_sql_query(query, conn)
print("initial shape of the dataset:", df.shape)
df.head(10) # to see the first 10 items in our dataframe

initial shape of the dataset: (9878978, 10)


,species,boa,qai,dbh_mm,height_dm,crown_area_m2,time,tree_id,X_WGS84,Y_WGS84
0,10,b'\xda\x00`\x01\xeb\x00\x07\x02\xee\x05\xfc\x0...,8192,254,201,12.342376,1438732800,94895,9.268243,48.126625
1,10,b'\x01\x01~\x01\xf3\x00\x14\x02\xd0\x05\xef\x0...,8192,254,201,12.342376,1439251200,94895,9.268243,48.126625
2,10,"b'\x84\x00\xfe\x00\x83\x00\x8f\x01,\x05\x8e\x0...",8192,254,201,12.342376,1440460800,94895,9.268243,48.126625
3,10,b'r\x00\xe1\x00\x7f\x00d\x01\xbe\x04\xf2\x05x\...,8192,254,201,12.342376,1441065600,94895,9.268243,48.126625
4,10,b'\xd6\x00\xd7\x00\xcd\x00S\x01\x93\x03\x05\x0...,10240,254,201,12.342376,1449360000,94895,9.268243,48.126625
5,10,b'(\x00\x9c\x00g\x00B\x01T\x04{\x05X\x05\xe9\x...,10240,254,201,12.342376,1453680000,94895,9.268243,48.126625
6,10,b'\x95\x00-\x01\xc3\x00\xea\x01\xf3\x04\xe4\x0...,8192,254,201,12.342376,1462147200,94895,9.268243,48.126625
7,10,b'3\x01\xc0\x01\x11\x01?\x02=\x06=\x08\x9a\x08...,8192,254,201,12.342376,1466726400,94895,9.268243,48.126625
8,10,b'\x19\x01\x95\x01\x12\x01\x1c\x02P\x06/\x08\r...,8192,254,201,12.342376,1470441600,94895,9.268243,48.126625
9,10,b'n\x00\xe2\x00m\x00u\x01S\x05\xca\x06\xb8\x06...,8192,254,201,12.342376,1470960000,94895,9.268243,48.126625


The bottom of atmosphere (BOA) reflectances still look strange! That's because they are displayed as raw byte characters. We can easily convert the 20 byte values to 10 16 bit signed integers.

You can find an overview of the Sentinel-2 bands [here](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/bands/).

<table><tbody><tr><th>Band</th><th>Resolution</th><th>Central Wavelength</th><th>Description</th></tr><tr><td>B2</td><td>10 m</td><td>490 nm</td><td>Blue</td></tr><tr><td>B3</td><td>10 m</td><td>560 nm</td><td>Green</td></tr><tr><td>B4</td><td>10 m</td><td>665 nm</td><td>Red</td></tr><tr><td>B5</td><td>20 m</td><td>705 nm</td><td>Visible and Near Infrared (VNIR)</td></tr><tr><td>B6</td><td>20 m</td><td>740 nm</td><td>Visible and Near Infrared (VNIR)</td></tr><tr><td>B7</td><td>20 m</td><td>783 nm</td><td>Visible and Near Infrared (VNIR)</td></tr><tr><td>B8</td><td>10 m</td><td>842 nm</td><td>Near Infrared (VNIR)</td></tr><tr><td>B8a</td><td>20 m</td><td>865 nm</td><td>Near Infrared (VNIR)</td></tr><tr><td>B11</td><td>20 m</td><td>1610 nm</td><td>Short Wave Infrared (SWIR)</td></tr><tr><td>B12</td><td>20 m</td><td>2190 nm</td><td>Short Wave Infrared (SWIR)</td></tr></tbody></table>

In [5]:
df.boa = [np.frombuffer(x, dtype=np.int16) for x in df.boa]
df.head(10)

,species,boa,qai,dbh_mm,height_dm,crown_area_m2,time,tree_id,X_WGS84,Y_WGS84
0,10,"[218, 352, 235, 519, 1518, 1788, 1939, 1971, 9...",8192,254,201,12.342376,1438732800,94895,9.268243,48.126625
1,10,"[257, 382, 243, 532, 1488, 1775, 1950, 2131, 8...",8192,254,201,12.342376,1439251200,94895,9.268243,48.126625
2,10,"[132, 254, 131, 399, 1324, 1678, 1783, 1889, 6...",8192,254,201,12.342376,1440460800,94895,9.268243,48.126625
3,10,"[114, 225, 127, 356, 1214, 1522, 1656, 1682, 6...",8192,254,201,12.342376,1441065600,94895,9.268243,48.126625
4,10,"[214, 215, 205, 339, 915, 1285, 1156, 1285, 36...",10240,254,201,12.342376,1449360000,94895,9.268243,48.126625
5,10,"[40, 156, 103, 322, 1108, 1403, 1368, 1513, 38...",10240,254,201,12.342376,1453680000,94895,9.268243,48.126625
6,10,"[149, 301, 195, 490, 1267, 1508, 1493, 1647, 7...",8192,254,201,12.342376,1462147200,94895,9.268243,48.126625
7,10,"[307, 448, 273, 575, 1597, 2109, 2202, 2231, 8...",8192,254,201,12.342376,1466726400,94895,9.268243,48.126625
8,10,"[281, 405, 274, 540, 1616, 2095, 2061, 2324, 8...",8192,254,201,12.342376,1470441600,94895,9.268243,48.126625
9,10,"[110, 226, 109, 373, 1363, 1738, 1720, 1939, 7...",8192,254,201,12.342376,1470960000,94895,9.268243,48.126625


Already looks much better! But what about the time?! Let's convert it from unix time to a human readable format.

In [6]:
df.time = pd.to_datetime(df.time, unit="s", utc=True)
df.head(10)

,species,boa,qai,dbh_mm,height_dm,crown_area_m2,time,tree_id,X_WGS84,Y_WGS84
0,10,"[218, 352, 235, 519, 1518, 1788, 1939, 1971, 9...",8192,254,201,12.342376,2015-08-05 00:00:00+00:00,94895,9.268243,48.126625
1,10,"[257, 382, 243, 532, 1488, 1775, 1950, 2131, 8...",8192,254,201,12.342376,2015-08-11 00:00:00+00:00,94895,9.268243,48.126625
2,10,"[132, 254, 131, 399, 1324, 1678, 1783, 1889, 6...",8192,254,201,12.342376,2015-08-25 00:00:00+00:00,94895,9.268243,48.126625
3,10,"[114, 225, 127, 356, 1214, 1522, 1656, 1682, 6...",8192,254,201,12.342376,2015-09-01 00:00:00+00:00,94895,9.268243,48.126625
4,10,"[214, 215, 205, 339, 915, 1285, 1156, 1285, 36...",10240,254,201,12.342376,2015-12-06 00:00:00+00:00,94895,9.268243,48.126625
5,10,"[40, 156, 103, 322, 1108, 1403, 1368, 1513, 38...",10240,254,201,12.342376,2016-01-25 00:00:00+00:00,94895,9.268243,48.126625
6,10,"[149, 301, 195, 490, 1267, 1508, 1493, 1647, 7...",8192,254,201,12.342376,2016-05-02 00:00:00+00:00,94895,9.268243,48.126625
7,10,"[307, 448, 273, 575, 1597, 2109, 2202, 2231, 8...",8192,254,201,12.342376,2016-06-24 00:00:00+00:00,94895,9.268243,48.126625
8,10,"[281, 405, 274, 540, 1616, 2095, 2061, 2324, 8...",8192,254,201,12.342376,2016-08-06 00:00:00+00:00,94895,9.268243,48.126625
9,10,"[110, 226, 109, 373, 1363, 1738, 1720, 1939, 7...",8192,254,201,12.342376,2016-08-12 00:00:00+00:00,94895,9.268243,48.126625


## Quality Filtering

First, lets keep only tree samples that come from summer months. I know this data comes from Germany, so all latitudes are positive and summer corresponds to June-August but let's keep our filter general and suppose we might have had data from the Southern hemisphere as well.

In [7]:
df = df[
    ((df.Y_WGS84 >= 0) & df.time.dt.month.between(6, 8)) |
    ((df.Y_WGS84 < 0) & df.time.dt.month.isin([12, 1, 2]))
]
# We will keep track of the day of year now and let go of the full time.
df["doy"] = df.time.dt.dayofyear
df = df.drop(columns=["time"])
print("shape of the dataset after seasonal filtering:", df.shape)
print(df.head(10))

shape of the dataset after seasonal filtering: (3146686, 10)
    species                                                boa   qai  dbh_mm  \
0        10  [218, 352, 235, 519, 1518, 1788, 1939, 1971, 9...  8192     254   
1        10  [257, 382, 243, 532, 1488, 1775, 1950, 2131, 8...  8192     254   
2        10  [132, 254, 131, 399, 1324, 1678, 1783, 1889, 6...  8192     254   
7        10  [307, 448, 273, 575, 1597, 2109, 2202, 2231, 8...  8192     254   
8        10  [281, 405, 274, 540, 1616, 2095, 2061, 2324, 8...  8192     254   
9        10  [110, 226, 109, 373, 1363, 1738, 1720, 1939, 7...  8192     254   
10       10  [76, 198, 102, 347, 1293, 1585, 1735, 1796, 62...  8192     254   
12       10  [173, 260, 177, 391, 1289, 1575, 1784, 1854, 6...  8192     254   
27       10  [160, 324, 167, 530, 1548, 1732, 1879, 1860, 7...  8192     254   
28       10  [212, 393, 218, 550, 1777, 2033, 2187, 2191, 8...  8192     254   

    height_dm  crown_area_m2  tree_id   X_WGS84    Y_WGS84

As you might have noticed, there is a QAI column, which stands for quality assurance information. This information is output by the [FORCE](https://force-eo.readthedocs.io/) satellite image processing suite. The different image conditions are encoded bit-wise in a single unsigned 16 bit integer, as documented [here](https://force-eo.readthedocs.io/en/latest/howto/qai.html#quality-bits-in-force).

For convenience, here are the different bit-masks in python code:

In [8]:
VALID        = 0b000000000000000
NODATA       = 0b000000000000001
CLOUD_BUFFER = 0b000000000000010
CLOUD_OPAQUE = 0b000000000000100
CLOUD_CIRRUS = 0b000000000000110
CLOUD_SHADOW = 0b000000000001000
SNOW         = 0b000000000010000
WATER        = 0b000000000100000
AOD_INT      = 0b000000001000000
AOD_HIGH     = 0b000000010000000
AOD_FILL     = 0b000000011000000
SUBZERO      = 0b000000100000000
SATURATION   = 0b000001000000000
SUN_LOW      = 0b000010000000000
ILLUMIN_LOW  = 0b000100000000000
ILLUMIN_POOR = 0b001000000000000
ILLUMIN_NONE = 0b001100000000000
SLOPED       = 0b010000000000000
WVP_NONE     = 0b100000000000000

Let's now look at the QAI value of the a record:

In [9]:
df.iloc[0]["qai"]

8192

...and convert this value to binary representation:

In [10]:
bin(df.iloc[0]["qai"])

'0b10000000000000'

Now we can count the position of the 1, starting from the right - the 1 is in position 14. Comparing this with the table above tells us, that the area is sloped. Let's check that:

In [11]:
SLOPED

8192

Correct!

Let's perform a simple check: Are there any pixels in our dataset that FORCE detected as water?

In [12]:
my_filter = WATER

If we now want to find out whether some of the QAI values matches our filter, we simply have to check whether there is a 1 in the same positions as in our filter. We do this by the "logical and" operator "&" that sets a bit if the QAI value *and* our filter have a bit set in the same position. Naturally, any bit set to 1 will yield a number larger zero.

In [13]:
mask = (df["qai"] & my_filter) > 0
mask

0          False
1          False
2          False
7          False
8          False
           ...  
9878971    False
9878972    False
9878973    False
9878974    False
9878975    False
Name: qai, Length: 3146686, dtype: bool

In [14]:
np.sum(mask)

168

-> There are 168 samples in this subset of the data that we have loaded where FORCE recognized water. Water of course makes no sense, because the measured trees are on land, but is obvousily possible in rainy conditions.

Let's now remove all types of cloud and snow and other poor conditions from our dataset by comparing each QAI value with a filter by the logical and operator. If the QAI values and the filter have no common bits the resulting value is zero - these are the "clean" records we are interested in.

Note that we can combine the different bit flags using a "logical or" operator "|". "a | b" compares two values and sets a bit where either a or b has a bit set.

In [15]:
#Use logical or "|" to combine filter values, logical and "&" to compare the filter with the QAI flag and check whether the result is zero to include all valid records.
qa_filter = NODATA | CLOUD_BUFFER | CLOUD_CIRRUS | CLOUD_OPAQUE | CLOUD_SHADOW | SNOW | WATER | AOD_HIGH | AOD_FILL | AOD_INT | ILLUMIN_POOR | SUN_LOW | SATURATION
df = df[(df["qai"] & qa_filter) == 0]
print("shape of the dataset after quality filtering:", df.shape)

shape of the dataset after quality filtering: (2653641, 10)


Let's drop the quality assurance column as we don't need it for classification anymore.

In [16]:
df = df.drop(columns=["qai"])

Sentinel-2 can produce negative reflectance values over trees, particularly in Level-2A (bottom-of-atmosphere) products. However, it is physically impossible. Negative values occur due to artifacts in atmospheric correction over very dark targets, such as deep shadows in dense, coniferous forests or water bodies, where the correction algorithm overcompensates for atmospheric scattering. So, it is best to detect and remove them from our dataset too.

In [17]:
df = df[df["boa"].apply(lambda x: min(x) >= 0)]
print("shape of the dataset after reflectance filtering:", df.shape)

shape of the dataset after reflectance filtering: (2644231, 9)


## NDVI

To explore the data even more, we can take a look at the Normalized Difference Vegetation Index (NDVI). We define the NDVI via the red and near infrared bands. We select the 842nm NIR band, because it has 10m resolution, like the red band.

In [18]:
def ndvi(boa):
    boa32 = boa.astype(np.float32)
    return (boa32[6] - boa32[2]) / (boa32[6] + boa32[2] + np.float32(1e-8))

In [19]:
df["ndvi"] = df["boa"].apply(ndvi)
df.head(10)

,species,boa,dbh_mm,height_dm,crown_area_m2,tree_id,X_WGS84,Y_WGS84,doy,ndvi
0,10,"[218, 352, 235, 519, 1518, 1788, 1939, 1971, 9...",254,201,12.342376,94895,9.268243,48.126625,217,0.783809
1,10,"[257, 382, 243, 532, 1488, 1775, 1950, 2131, 8...",254,201,12.342376,94895,9.268243,48.126625,223,0.778386
2,10,"[132, 254, 131, 399, 1324, 1678, 1783, 1889, 6...",254,201,12.342376,94895,9.268243,48.126625,237,0.863114
7,10,"[307, 448, 273, 575, 1597, 2109, 2202, 2231, 8...",254,201,12.342376,94895,9.268243,48.126625,176,0.779394
8,10,"[281, 405, 274, 540, 1616, 2095, 2061, 2324, 8...",254,201,12.342376,94895,9.268243,48.126625,219,0.765310
9,10,"[110, 226, 109, 373, 1363, 1738, 1720, 1939, 7...",254,201,12.342376,94895,9.268243,48.126625,225,0.880809
10,10,"[76, 198, 102, 347, 1293, 1585, 1735, 1796, 62...",254,201,12.342376,94895,9.268243,48.126625,236,0.888949
12,10,"[173, 260, 177, 391, 1289, 1575, 1784, 1854, 6...",254,201,12.342376,94895,9.268243,48.126625,244,0.819480
27,10,"[160, 324, 167, 530, 1548, 1732, 1879, 1860, 7...",254,201,12.342376,94895,9.268243,48.126625,153,0.836755
28,10,"[212, 393, 218, 550, 1777, 2033, 2187, 2191, 8...",254,201,12.342376,94895,9.268243,48.126625,170,0.818711


Since we already removed outliers from our dataset, NDVI values should be bound to -1 and 1, but let's do a final check, and remove any samples
that might have NaN or out-of-bound values as NDVI.

In [20]:
df = df[df["ndvi"].between(-1, 1) & df["ndvi"].notna()]
print("shape of the dataset after ndvi filtering:", df.shape)

shape of the dataset after ndvi filtering: (2644231, 10)


## Train/Test Splitting and Storing the Dataset

Now let's already split our data to train, val, test for future classification task. 

*Why not random split?*

If you split rows randomly, you risk:

- Tree leakage: same tree appears in train and test via multiple samples (inflates metrics).
- Spatial leakage: training contains trees very close to test trees (spatial autocorrelation), inflating metrics; blocked spatial strategies reduce that bias.

So, we group the samples by tree_id and by their geographic proximity, and then assign these "groups" randomly to the train/test splits based on a percentage, e.g. 70% train, 30% test.

In [21]:
def split_by_tree_geo(df):
    # tree-level table
    tree_tbl = (
        df.groupby("tree_id", as_index=False)
        .agg(lon=("X_WGS84", "mean"), lat=("Y_WGS84", "mean"), y=("species", "first"))
    )

    # Germany is primarily in the UTM zone 32N (EPSG code 25832) for its mapping and surveying purposes.
    gdf = gpd.GeoDataFrame(
        tree_tbl,
        geometry=gpd.points_from_xy(tree_tbl["lon"], tree_tbl["lat"]),
        crs="EPSG:4326"  # WGS84 (degrees)
    )
    
    # Germany-friendly projected CRS (meters)
    gdf = gdf.to_crs("EPSG:25832")  # ETRS89 / UTM zone 32N
    
    tree_tbl["x_m"] = gdf.geometry.x
    tree_tbl["y_m"] = gdf.geometry.y

    BLOCK_SIZE_M = 20000.0  # 20 km blocks (something I decided, but depending on our area, we could change it to be more meaninful)
    tree_tbl["bx"] = np.floor(tree_tbl["x_m"] / BLOCK_SIZE_M).astype(int)
    tree_tbl["by"] = np.floor(tree_tbl["y_m"] / BLOCK_SIZE_M).astype(int)
    tree_tbl["block_id"] = (tree_tbl["bx"].astype(str) + "_" + tree_tbl["by"].astype(str))

    # split blocks (groups = block_id)
    block_gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    tr_tree_idx, te_tree_idx = next(block_gss.split(tree_tbl, tree_tbl["y"], groups=tree_tbl["block_id"]))

    tree_tr = tree_tbl.iloc[tr_tree_idx].copy()
    tree_te = tree_tbl.iloc[te_tree_idx].copy()

    # split validation blocks inside train blocks
    block_gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    tr2_idx, va_idx = next(block_gss2.split(tree_tr, tree_tr["y"], groups=tree_tr["block_id"]))
    tree_va = tree_tr.iloc[va_idx].copy()
    tree_tr = tree_tr.iloc[tr2_idx].copy()

    train_tree_ids = set(tree_tr["tree_id"])
    val_tree_ids = set(tree_va["tree_id"])
    test_tree_ids = set(tree_te["tree_id"])

    df["split"] = "unused"

    df.loc[df.tree_id.isin(train_tree_ids), "split"] = "train"
    df.loc[df.tree_id.isin(val_tree_ids), "split"] = "val"
    df.loc[df.tree_id.isin(test_tree_ids), "split"] = "test"

    df_train = df[df.split == "train"]
    df_val   = df[df.split == "val"]
    df_test  = df[df.split == "test"]

    print("train, validtaion, test size")
    print(df_train.shape, df_val.shape, df_test.shape)

    print("train, validation and test unique tree_id counts: ")
    print(len(set(tree_tr.tree_id)), len(set(tree_va.tree_id)), len(set(tree_te.tree_id)))

    print("Tree overlap train/test:", len(set(df_train.tree_id) & set(df_test.tree_id)))
    print("Tree overlap train/val:", len(set(df_train.tree_id) & set(df_val.tree_id)))

    print("train, validation and test unique block_id counts: ")
    print(len(set(tree_tr.block_id)), len(set(tree_va.block_id)), len(set(tree_te.block_id)))
    print("Geographic overlap train/test:", len(set(tree_tr.block_id) & set(tree_te.block_id)))
    print("Geographic overlap train/val:", len(set(tree_tr.block_id) & set(tree_va.block_id)))


In [22]:
split_by_tree_geo(df)
df.head()

train, validtaion, test size
(1574978, 11) (343413, 11) (725840, 11)
train, validation and test unique tree_id counts: 
30306 7070 13766
Tree overlap train/test: 0
Tree overlap train/val: 0
train, validation and test unique block_id counts: 
417 105 225
Geographic overlap train/test: 0
Geographic overlap train/val: 0


,species,boa,dbh_mm,height_dm,crown_area_m2,tree_id,X_WGS84,Y_WGS84,doy,ndvi,split
0,10,"[218, 352, 235, 519, 1518, 1788, 1939, 1971, 9...",254,201,12.342376,94895,9.268243,48.126625,217,0.783809,test
1,10,"[257, 382, 243, 532, 1488, 1775, 1950, 2131, 8...",254,201,12.342376,94895,9.268243,48.126625,223,0.778386,test
2,10,"[132, 254, 131, 399, 1324, 1678, 1783, 1889, 6...",254,201,12.342376,94895,9.268243,48.126625,237,0.863114,test
7,10,"[307, 448, 273, 575, 1597, 2109, 2202, 2231, 8...",254,201,12.342376,94895,9.268243,48.126625,176,0.779394,test
8,10,"[281, 405, 274, 540, 1616, 2095, 2061, 2324, 8...",254,201,12.342376,94895,9.268243,48.126625,219,0.765310,test


We now save this cleaned dataframe to a paqruet file for future use. A Parquet file is an Apache open-source, column-oriented binary file format designed for efficient data storage and retrieval in big data analytics. Unlike traditional row-based formats like CSV or JSON, which store data row by row, Parquet groups data by column, leading to significant performance and storage benefits (the CSV version of this dataset would take up 350 MB space on disk while Parquet version would need 50 MB).

In [23]:
save_path = "trees_dataset.parquet"
df.to_parquet(
    save_path,
    engine="pyarrow",
    compression="snappy",
    index=False,
    row_group_size=500_000
)

When working with parquets, there is a way to lazy-load them to memory and stream instead of full loading to memory.

In [24]:
lf = pl.scan_parquet(save_path)

boa_labels = ["Blue", "Green", "Red", "B05", "B06", "B07", "NIR", "B8a", "SWIR1", "SWIR2"]

boa_cols = [pl.col("boa").list.get(i).cast(pl.Int32).alias(boa_labels[i]) for i in range(10)]
lf = lf.with_columns(boa_cols).drop("boa")

lf = lf.with_columns([
    pl.col("species").cast(pl.Int32),
    pl.col("tree_id").cast(pl.Int32),
    pl.col("dbh_mm").cast(pl.Float32),
    pl.col("height_dm").cast(pl.Float32),
    pl.col("crown_area_m2").cast(pl.Float32),
    pl.col("X_WGS84").cast(pl.Float32),
    pl.col("Y_WGS84").cast(pl.Float32),
    pl.col("doy").cast(pl.Int32),
    pl.col("ndvi").cast(pl.Float32),
    pl.col("split").cast(pl.String)
])

#this helps with later grouping by tree_id (we will see in next demo)
lf = (
    lf
    .sort("tree_id")
    .select(pl.all())  # forces column materialization
)

lf.sink_parquet(
    save_path,
    compression="zstd"
)

# Peek schema
lf.collect_schema()


Schema([('species', Int32),
        ('dbh_mm', Float32),
        ('height_dm', Float32),
        ('crown_area_m2', Float32),
        ('tree_id', Int32),
        ('X_WGS84', Float32),
        ('Y_WGS84', Float32),
        ('doy', Int32),
        ('ndvi', Float32),
        ('split', String),
        ('Blue', Int32),
        ('Green', Int32),
        ('Red', Int32),
        ('B05', Int32),
        ('B06', Int32),
        ('B07', Int32),
        ('NIR', Int32),
        ('B8a', Int32),
        ('SWIR1', Int32),
        ('SWIR2', Int32)])